In [1]:
import requests
import pandas as pd
import numpy as np
from datetime import datetime
import FinanceDataReader as fdr
import time
from typing import List, Dict
from typing import Optional
import io
from DATA.stock_invest_function import get_db_host

start_dt = '2025-06-01'

def fetch_fmp_price(symbol: str,
                    api_key: str,
                    start_date: str = start_dt) -> pd.DataFrame:
    """
    FMP historical-price-full API로 일별 가격 (Adj Close 중심) 수집

    Returns
    -------
    DataFrame: index=date, columns=[open, high, low, close, adjClose, volume]
    """
    url = f"https://financialmodelingprep.com/api/v3/historical-price-full/{symbol}"
    params = {
        "from": start_date,
        "apikey": api_key,
        "serietype": "line"   # adjClose 중심
    }
    r = requests.get(url, params=params)
    r.raise_for_status()
    data = r.json()

    if "historical" not in data or len(data["historical"]) == 0:
        raise ValueError(f"{symbol}: 가격 데이터 없음")

    df = pd.DataFrame(data["historical"])
    df["date"] = pd.to_datetime(df["date"])
    df = df.set_index("date").sort_index()

    # 컬럼 정리 (adjClose가 없으면 close 사용)
    if "adjClose" not in df.columns:
        df["adjClose"] = df["close"]

    cols = ["open", "high", "low", "close", "adjClose", "volume"]
    df = df[[c for c in cols if c in df.columns]]

    return df

def calc_daily_returns_and_beta(
    price_stock: pd.DataFrame,
    price_mkt: pd.DataFrame,
    windows: List[int] = [252, 750, 1250]
) -> pd.DataFrame:
    """
    종목/시장지수 가격으로 일간 수익률 및 rolling 베타 계산

    Returns
    -------
    DataFrame index=date,
    columns=['ret_stock', 'ret_mkt', 'beta_252', 'beta_750', 'beta_1250', ...]
    """
    df = pd.DataFrame(index=price_stock.index.union(price_mkt.index))
    df = df.sort_index()

    df["price_stock"] = price_stock["adjClose"]
    df["price_mkt"] = price_mkt["adjClose"]
    df = df.dropna()

    df["ret_stock"] = df["price_stock"].pct_change()
    df["ret_mkt"]   = df["price_mkt"].pct_change()
    df = df.dropna(subset=["ret_stock", "ret_mkt"])

    for w in windows:
        cov = df["ret_stock"].rolling(w).cov(df["ret_mkt"])
        var = df["ret_mkt"].rolling(w).var()
        df[f"beta_{w}"] = cov / var

    return df

def fetch_us_treasury_yields(
    start_date: str = start_dt,
    end_date: Optional[str] = None
) -> pd.DataFrame:
    """
    FinanceDataReader를 사용해 1Y, 3Y, 5Y 미국 국채 금리(연율, %)를 불러온 뒤
    소수(0.x) 단위 연율로 반환.
    """
    if end_date is None:
        end_date = pd.Timestamp.today().strftime("%Y-%m-%d")

    y1 = fdr.DataReader("FRED:DGS1", start_date, end_date)
    y3 = fdr.DataReader("FRED:DGS3", start_date, end_date)
    y5 = fdr.DataReader("FRED:DGS5", start_date, end_date)

    df = pd.DataFrame(index=y1.index.union(y3.index).union(y5.index)).sort_index()
    df["rf_1y"] = y1.iloc[:, 0] / 100.0   # % → 소수
    df["rf_3y"] = y3.iloc[:, 0] / 100.0
    df["rf_5y"] = y5.iloc[:, 0] / 100.0

    return df



def _fetch_fred_series(series_id: str,
                       start_date: str = start_dt,
                       end_date: Optional[str] = None) -> pd.DataFrame:
    """
    FRED CSV를 직접 호출해서 단일 시계열(series_id)을 DataFrame으로 반환.
    예: series_id = 'DGS1', 'DGS3', 'DGS5'
    """
    if end_date is None:
        end_date = pd.Timestamp.today().strftime("%Y-%m-%d")

    url = f"https://fred.stlouisfed.org/graph/fredgraph.csv?id={series_id}"
    r = requests.get(url)
    r.raise_for_status()

    # CSV 파싱
    df = pd.read_csv(io.StringIO(r.text))

    # FRED 포맷은 보통 DATE / <series_id> 이런 식
    if "DATE" not in df.columns:
        raise ValueError(f"FRED 응답에 DATE 컬럼이 없습니다. series_id={series_id}")

    df["DATE"] = pd.to_datetime(df["DATE"])
    df = df.set_index("DATE").sort_index()

    # '.' 등 결측 기호를 NaN으로 변환 후 float 처리
    col = df.columns[0]
    df[col] = df[col].replace(".", np.nan).astype(float)

    # 기간 필터링
    df = df.loc[start_date:end_date]

    return df

def fetch_us_treasury_yields(
    start_date: str = start_dt,
    end_date: Optional[str] = None
) -> pd.DataFrame:
    """
    1순위: FDR(FRED:DGS1, DGS3, DGS5)
    2순위: Yahoo Finance (^IRX=3M, ^FVX=5Y, ^TNX=10Y)에서 근사
    ※ 금리는 '그대로' 사용. 연율화/조정 없음.
    """
    if end_date is None:
        end_date = pd.Timestamp.today().strftime("%Y-%m-%d")

    # ====== 1) FDR(FRED) 시도 ======
    try:
        import FinanceDataReader as fdr
        y1 = fdr.DataReader("FRED:DGS1", start_date, end_date)
        y3 = fdr.DataReader("FRED:DGS3", start_date, end_date)
        y5 = fdr.DataReader("FRED:DGS5", start_date, end_date)

        df = pd.DataFrame(index=y1.index.union(y3.index).union(y5.index)).sort_index()
        df["rf_1y"] = y1.iloc[:, 0] / 100.0
        df["rf_3y"] = y3.iloc[:, 0] / 100.0
        df["rf_5y"] = y5.iloc[:, 0] / 100.0

        print("[INFO] FDR(FRED) 금리 사용")
        return df

    except Exception as e:
        print(f"[WARN] FDR 실패: {e}")
        print("[INFO] Yahoo Finance로 fallback합니다")

    # ====== 2) Yahoo Finance fallback ======
    import FinanceDataReader as fdr

    # 1년물 근사 = 1-year T-bill proxy (^IRX: 13-week but annualized → 그대로 소수 변환)
    y1 = fdr.DataReader("^IRX", start_date, end_date)

    # 5년물 (^FVX) 직접 사용
    y5 = fdr.DataReader("^FVX", start_date, end_date)

    idx = y1.index.union(y5.index)
    df = pd.DataFrame(index=idx).sort_index()

    df["rf_1y"] = y1["Close"] / 100.0     # 연율 금리 그대로 소수 변환
    df["rf_5y"] = y5["Close"] / 100.0

    # 3년물 Yahoo Finance 없음 → 1Y~5Y 간 선형보간 (경제적으로 자연스러움)
    df["rf_3y"] = df["rf_1y"] + (df["rf_5y"] - df["rf_1y"]) * (3 - 1) / (5 - 1)

    return df


def compute_capm_required_returns(
    df: pd.DataFrame,
    rf_df: pd.DataFrame
) -> pd.DataFrame:
    """
    df: calc_daily_returns_and_beta 결과 (ret_stock, ret_mkt, beta_252, beta_750, beta_1250 포함)
    rf_df: fetch_us_treasury_yields 결과 (rf_1y, rf_3y, rf_5y)

    Returns
    -------
    df_capm: df에 E_Rm_1y,3y,5y 및 Re_1y,3y,5y(연율)를 추가한 DataFrame
    """
    out = df.copy()

    # 국채금리 merge (forward fill)
    rf_df = rf_df.copy()
    rf_df = rf_df.reindex(out.index).sort_index()
    rf_df = rf_df.ffill()

    out[["rf_1y", "rf_3y", "rf_5y"]] = rf_df[["rf_1y", "rf_3y", "rf_5y"]]

    # 시장 수익률 rolling 평균 (연율화)
    out["E_Rm_1y"] = out["ret_mkt"].rolling(252).mean() * 252
    out["E_Rm_3y"] = out["ret_mkt"].rolling(750).mean() * 252
    out["E_Rm_5y"] = out["ret_mkt"].rolling(1250).mean() * 252

    # CAPM 필요수익률 (연율)
    out["Re_1y"] = out["rf_1y"] + out["beta_252"]  * (out["E_Rm_1y"] - out["rf_1y"])
    out["Re_3y"] = out["rf_3y"] + out["beta_750"]  * (out["E_Rm_3y"] - out["rf_3y"])
    out["Re_5y"] = out["rf_5y"] + out["beta_1250"] * (out["E_Rm_5y"] - out["rf_5y"])

    return out

def run_capm_pipeline_for_ticker(
    ticker: str,
    api_key: str,
    market_symbol: str = "^GSPC",
    start_date: str = start_dt
) -> pd.DataFrame:
    """
    1개 종목에 대해:
      - FMP 가격 (종목, 시장지수)
      - 일간 수익률/베타 (1Y=252, 3Y=750, 5Y=1250)
      - FDR 국채금리(1,3,5Y)
      - CAPM 필요수익률(연율) Re_1y, Re_3y, Re_5y

    Returns
    -------
    DataFrame index=date
    columns = [
       'price_stock', 'price_mkt',
       'ret_stock', 'ret_mkt',
       'beta_252','beta_750','beta_1250',
       'rf_1y','rf_3y','rf_5y',
       'E_Rm_1y','E_Rm_3y','E_Rm_5y',
       'Re_1y','Re_3y','Re_5y'
    ]
    """
    print(f"[INFO] {ticker} 가격 데이터 수집 (FMP)...")
    price_stock = fetch_fmp_price(ticker, api_key, start_date)
    print(f"[INFO] {market_symbol} 지수 데이터 수집 (FMP)...")
    price_mkt   = fetch_fmp_price(market_symbol, api_key, start_date)

    print("[INFO] 일간 수익률 및 rolling 베타 계산...")
    base_df = calc_daily_returns_and_beta(price_stock, price_mkt)

    print("[INFO] 미국 국채금리(1Y,3Y,5Y) 수집 (FDR/Yahoo fallback)...")
    rf_df = fetch_us_treasury_yields(
        start_date=start_date,
        end_date=base_df.index.max().strftime("%Y-%m-%d")
    )

    print("[INFO] CAPM 필요수익률 계산...")
    result_df = compute_capm_required_returns(base_df, rf_df)

    return result_df

# get_filtered_us_tickers.py
# 미국 NASDAQ, NYSE, AMEX 상장사 중 특정 IndustryCode 앞 4자리를 제외한 ticker 리스트 반환

import FinanceDataReader as fdr

def get_filtered_us_tickers():
    """
    NASDAQ / NYSE / AMEX 전체 상장사 정보를 기반으로
    특정 IndustryCode 앞 4자리에 해당하는 기업들을 제외하고
    최종 ticker 리스트를 반환하는 함수.

    Returns:
        tickers (list): 필터링된 미국 상장사 ticker 리스트
    """

    # 1) 미국 주요 거래소 전체 티커 로드
    nasdaq = fdr.StockListing('NASDAQ')
    nyse = fdr.StockListing('NYSE')
    amex = fdr.StockListing('AMEX')

    # 하나의 DF로 통합
    info_df = pd.concat([nasdaq, nyse, amex], ignore_index=True)

    # 2) 제외할 IndustryCode 앞 4자리 목록
    exclude_prefixes = ["5510", "5730", "5530", "6010", "5910", "5120"]

    # 3) IndustryCode 정리
    info_df["IndustryCode"] = info_df["IndustryCode"].astype(str)
    info_df["IndustryPrefix"] = info_df["IndustryCode"].str[:4]

    # 4) 제외 마스크 생성
    mask_exclude = info_df["IndustryPrefix"].isin(exclude_prefixes)

    # 5) 제외된 기업 / 남은 기업 구분
    excluded_df = info_df[mask_exclude]
    filtered_df = info_df[~mask_exclude].copy()

    # 6) 최종 ticker 리스트 추출
    tickers = filtered_df["Symbol"].unique().tolist()

    print(f"제외된 기업 수: {len(excluded_df)}")
    print(f"남은 기업 수: {len(filtered_df)}")
    print(f"티커 수: {len(tickers)}")

    return tickers

def results_to_long(results: Dict[str, pd.DataFrame],
                    indicator_prefixes: List[str] = ("beta_", "Re_")) -> pd.DataFrame:
    """
    results: {ticker: df_t} 형태 (df_t.index = date)
    indicator_prefixes: 컬럼명 접두어 (예: beta_*, Re_* 만 long 으로 뽑기)

    return: columns = [date, ticker, indicator, value]
    """
    long_list = []

    for ticker, df in results.items():
        if df.index.name != "date":
            df = df.copy()
            df.index.name = "date"
        df_tmp = df.copy()

        # 사용할 indicator 컬럼들만 선택
        use_cols = [c for c in df_tmp.columns
                    if any(c.startswith(pref) for pref in indicator_prefixes)]

        if not use_cols:
            print(f"[WARN] {ticker}: indicator 컬럼(beta_/Re_)이 없습니다. 스킵.")
            continue

        df_tmp = df_tmp[use_cols].reset_index()  # date를 컬럼으로

        df_long = df_tmp.melt(
            id_vars="date",
            value_vars=use_cols,
            var_name="indicator",
            value_name="value"
        )
        df_long["ticker"] = ticker

        # 컬럼 순서 정리
        df_long = df_long[["date", "ticker", "indicator", "value"]]

        # NaN 값 제거
        df_long = df_long.dropna(subset=["value"])

        long_list.append(df_long)

    if not long_list:
        raise ValueError("results_to_long: 변환할 데이터가 없습니다.")

    long_df = pd.concat(long_list, ignore_index=True)
    long_df["date"] = pd.to_datetime(long_df["date"])

    return long_df

def create_us_beta_re_table_if_not_exists(db_info: Dict[str, any],
                                          table_name: str = "us_fs_data_roe_roa") -> None:
    """
    us_fs_data_roe_roa 테이블 없으면 생성
    structure: date, ticker, indicator, value (PRIMARY KEY: date, ticker, indicator)
    """
    conn = pymysql.connect(
        host=db_info["host"],
        port=db_info.get("port", 3306),
        user=db_info["user"],
        password=db_info["password"],
        database=db_info["database"],
        charset="utf8mb4",
        autocommit=True
    )

    try:
        with conn.cursor() as cur:
            create_sql = f"""
            CREATE TABLE IF NOT EXISTS {table_name} (
                date DATE NOT NULL,
                ticker VARCHAR(20) NOT NULL,
                indicator VARCHAR(50) NOT NULL,
                value DOUBLE,
                PRIMARY KEY (date, ticker, indicator)
            ) ENGINE=InnoDB DEFAULT CHARSET=utf8mb4;
            """
            cur.execute(create_sql)
            print(f"[INFO] 테이블 확인/생성 완료: {table_name}")
    finally:
        conn.close()


def upsert_us_beta_re_long_df(long_df: pd.DataFrame,
                              db_info: Dict[str, any],
                              table_name: str = "us_fs_data_roe_roa",
                              batch_size_rows: int = 50_000,
                              batch_size_ticker: int = 30) -> None:
    """
    long_df: columns = [date, ticker, indicator, value]
    - ticker를 30개씩 잘라서 저장
    - 각 batch 안에서는 rows를 batch_size_rows 단위로 나눠 INSERT ... ON DUPLICATE KEY UPDATE
    """
    # 테이블 없으면 생성
    create_us_beta_re_table_if_not_exists(db_info, table_name)

    # 티커별 배치 나누기
    tickers = sorted(long_df["ticker"].unique())
    n = len(tickers)

    print(f"[INFO] 총 티커 수: {n}, batch_size_ticker = {batch_size_ticker}")

    for i in range(0, n, batch_size_ticker):
        batch_tickers = tickers[i:i + batch_size_ticker]
        batch_df = long_df[long_df["ticker"].isin(batch_tickers)].copy()
        batch_df = batch_df.sort_values(["ticker", "date", "indicator"])

        print(f"[INFO] 티커 batch {i//batch_size_ticker+1}: "
              f"{len(batch_tickers)} tickers, {len(batch_df)} rows 저장 예정")

        if batch_df.empty:
            continue

        conn = pymysql.connect(
            host=db_info["host"],
            port=db_info.get("port", 3306),
            user=db_info["user"],
            password=db_info["password"],
            database=db_info["database"],
            charset="utf8mb4",
            autocommit=False
        )

        try:
            with conn.cursor() as cur:
                insert_sql = f"""
                INSERT INTO {table_name} (date, ticker, indicator, value)
                VALUES (%s, %s, %s, %s)
                ON DUPLICATE KEY UPDATE
                    value = VALUES(value)
                """

                rows = list(
                    batch_df[["date", "ticker", "indicator", "value"]]
                    .itertuples(index=False, name=None)
                )

                # row 수가 많을 수 있으니 chunk 로 나눠서 insert
                for j in range(0, len(rows), batch_size_rows):
                    chunk = rows[j:j + batch_size_rows]
                    cur.executemany(insert_sql, chunk)
                    conn.commit()
                    print(f"  - rows {j} ~ {j+len(chunk)-1} 커밋 완료")

        except Exception as e:
            conn.rollback()
            print(f"[ERROR] 티커 batch 저장 중 오류: {e}")
            raise
        finally:
            conn.close()

    print("[INFO] 모든 batch 저장 완료")


In [2]:
# 1) DB 설정
db_info = {
    "host": get_db_host(),
    "port": 3307,
    "user": "stox7412",
    "password": "Apt106503!~",   # 실제 비밀번호
    "database": "investar",
}

API_KEY = 'hT0gAk87j9xZx4PlBApvBqfVL5IahvgV'

# TICKER_LIST = ["AAPL", "MSFT", "NVDA"]  # 예시

TICKER_LIST = get_filtered_us_tickers()
# TICKER_LIST = TICKER_LIST[1000:2000]
# TICKER_LIST = ["BG", "PLMR", "LLY", "AMD", "DASH", "LULU", "CAT"]

100%|██████████| 299/299 [00:00<00:00, 525.03it/s]

제외된 기업 수: 1808
남은 기업 수: 4993
티커 수: 4993


In [3]:
# TICKER_LIST = TICKER_LIST[1500:2000]
# TICKER_LIST = ["APP"]

results: Dict[str, pd.DataFrame] = {}

for t in TICKER_LIST:
    try:
        df_t = run_capm_pipeline_for_ticker(
            ticker=t,
            api_key=API_KEY,
            market_symbol="^GSPC",     # S&P500 지수 기준
            start_date="2010-01-01"
        )
        results[t] = df_t
        print(f"[OK] {t} 처리 완료, {len(df_t)} rows")
        # FMP API 부담 줄이기 위해 살짝 대기
        time.sleep(1.0)
    except Exception as e:
        print(f"[ERROR] {t} 처리 중 오류: {e}")

[INFO] NVDA 가격 데이터 수집 (FMP)...
[INFO] ^GSPC 지수 데이터 수집 (FMP)...
[INFO] 일간 수익률 및 rolling 베타 계산...
[INFO] 미국 국채금리(1Y,3Y,5Y) 수집 (FDR/Yahoo fallback)...
[WARN] FDR 실패: Missing column provided to 'parse_dates': 'DATE'
[INFO] Yahoo Finance로 fallback합니다
[INFO] CAPM 필요수익률 계산...
[OK] NVDA 처리 완료, 4027 rows
[INFO] AAPL 가격 데이터 수집 (FMP)...
[INFO] ^GSPC 지수 데이터 수집 (FMP)...
[INFO] 일간 수익률 및 rolling 베타 계산...
[INFO] 미국 국채금리(1Y,3Y,5Y) 수집 (FDR/Yahoo fallback)...
[WARN] FDR 실패: Missing column provided to 'parse_dates': 'DATE'
[INFO] Yahoo Finance로 fallback합니다
[INFO] CAPM 필요수익률 계산...
[OK] AAPL 처리 완료, 4027 rows
[INFO] MSFT 가격 데이터 수집 (FMP)...
[INFO] ^GSPC 지수 데이터 수집 (FMP)...
[INFO] 일간 수익률 및 rolling 베타 계산...
[INFO] 미국 국채금리(1Y,3Y,5Y) 수집 (FDR/Yahoo fallback)...
[WARN] FDR 실패: Missing column provided to 'parse_dates': 'DATE'
[INFO] Yahoo Finance로 fallback합니다
[INFO] CAPM 필요수익률 계산...
[OK] MSFT 처리 완료, 4027 rows
[INFO] AMZN 가격 데이터 수집 (FMP)...
[INFO] ^GSPC 지수 데이터 수집 (FMP)...
[INFO] 일간 수익률 및 rolling 베타 계산...
[INFO] 미국 국채금

KeyboardInterrupt: 

In [4]:
import pymysql

# 1) DB 설정
db_info = {
    "host": get_db_host(),
    "port": 3307,
    "user": "stox7412",
    "password": "Apt106503!~",   # 실제 비밀번호
    "database": "investar",
}

# 2) CAPM 파이프라인 돌려서 results 만들기 (이미 하셨음)
# results = { "AAPL": df_aapl, "MSFT": df_msft, ... }

# 3) results → long format (beta_*, Re_*만)
long_df = results_to_long(results, indicator_prefixes=("beta_", "Re_"))
print(long_df.head())

# 4) 30개 티커씩 잘라서 DB에 저장
upsert_us_beta_re_long_df(
    long_df=long_df,
    db_info=db_info,
    table_name="us_required_return_result",
    batch_size_rows=50_000,
    batch_size_ticker=30
)

        date ticker indicator     value
0 2011-01-03     BG  beta_252  0.922316
1 2011-01-04     BG  beta_252  0.919633
2 2011-01-05     BG  beta_252  0.922025
3 2011-01-06     BG  beta_252  0.918603
4 2011-01-07     BG  beta_252  0.918761
[INFO] 테이블 확인/생성 완료: us_required_return_result
[INFO] 총 티커 수: 7, batch_size_ticker = 30
[INFO] 티커 batch 1: 7 tickers, 106520 rows 저장 예정
  - rows 0 ~ 49999 커밋 완료
  - rows 50000 ~ 99999 커밋 완료
  - rows 100000 ~ 106519 커밋 완료
[INFO] 모든 batch 저장 완료


In [25]:
results

{'NVDA':             price_stock  price_mkt  ret_stock   ret_mkt  beta_252  beta_750  \
 date                                                                          
 2010-01-05      0.46900    1136.52   0.014602  0.003116       NaN       NaN   
 2010-01-06      0.47200    1137.14   0.006397  0.000546       NaN       NaN   
 2010-01-07      0.46275    1141.69  -0.019597  0.004001       NaN       NaN   
 2010-01-08      0.46375    1144.98   0.002161  0.002882       NaN       NaN   
 2010-01-11      0.45725    1146.98  -0.014016  0.001747       NaN       NaN   
 ...                 ...        ...        ...       ...       ...       ...   
 2025-11-20    180.64000    6538.77  -0.031525 -0.015564  1.947900  2.174563   
 2025-11-21    178.88000    6602.98  -0.009743  0.009820  1.936223  2.170032   
 2025-11-24    182.55000    6705.11   0.020517  0.015467  1.932049  2.167842   
 2025-11-25    177.82000    6765.89  -0.025911  0.009065  1.922386  2.161876   
 2025-11-26    180.26000    6812

In [22]:
results: Dict[str, pd.DataFrame] = {}

for t in TICKER_LIST:
    try:
        df_t = run_capm_pipeline_for_ticker(
            ticker=t,
            api_key=API_KEY,
            market_symbol="^GSPC",     # S&P500 지수 기준
            start_date="2010-01-01"
        )
        results[t] = df_t
        print(f"[OK] {t} 처리 완료, {len(df_t)} rows")
        # FMP API 부담 줄이기 위해 살짝 대기
        time.sleep(1.0)
    except Exception as e:
        print(f"[ERROR] {t} 처리 중 오류: {e}")

[INFO] NVDA 가격 데이터 수집 (FMP)...
[INFO] ^GSPC 지수 데이터 수집 (FMP)...
[INFO] 일간 수익률 및 rolling 베타 계산...
[INFO] 미국 국채금리(1Y,3Y,5Y) 수집 (FDR/Yahoo fallback)...
[WARN] FDR 실패: Missing column provided to 'parse_dates': 'DATE'
[INFO] Yahoo Finance로 fallback합니다
[INFO] CAPM 필요수익률 계산...
[OK] NVDA 처리 완료, 4000 rows
[INFO] AAPL 가격 데이터 수집 (FMP)...
[INFO] ^GSPC 지수 데이터 수집 (FMP)...
[INFO] 일간 수익률 및 rolling 베타 계산...
[INFO] 미국 국채금리(1Y,3Y,5Y) 수집 (FDR/Yahoo fallback)...
[WARN] FDR 실패: Missing column provided to 'parse_dates': 'DATE'
[INFO] Yahoo Finance로 fallback합니다
[INFO] CAPM 필요수익률 계산...
[OK] AAPL 처리 완료, 4000 rows
[INFO] MSFT 가격 데이터 수집 (FMP)...
[INFO] ^GSPC 지수 데이터 수집 (FMP)...
[INFO] 일간 수익률 및 rolling 베타 계산...
[INFO] 미국 국채금리(1Y,3Y,5Y) 수집 (FDR/Yahoo fallback)...
[WARN] FDR 실패: Missing column provided to 'parse_dates': 'DATE'
[INFO] Yahoo Finance로 fallback합니다
[INFO] CAPM 필요수익률 계산...
[OK] MSFT 처리 완료, 4000 rows
[INFO] AMZN 가격 데이터 수집 (FMP)...
[INFO] ^GSPC 지수 데이터 수집 (FMP)...
[INFO] 일간 수익률 및 rolling 베타 계산...
[INFO] 미국 국채금

In [23]:
df_all = pd.concat(results, names=["AVGO"])


In [24]:
df_all.tail(20)

price_stock  price_mkt  ret_stock   ret_mkt  beta_252  \
AVGO date                                                                
NFLX 2025-10-30       108.90    6822.35  -0.010360 -0.009905  0.907965   
     2025-10-31       111.89    6840.19   0.027456  0.002615  0.908966   
     2025-11-03       110.01    6851.98  -0.016802  0.001724  0.907785   
     2025-11-04       109.30    6771.54  -0.006454 -0.011740  0.917004   
     2025-11-05       109.85    6796.30   0.005032  0.003656  0.917497   
     2025-11-06       109.70    6720.31  -0.001365 -0.011181  0.914936   
     2025-11-07       110.37    6728.81   0.006108  0.001265  0.915378   
     2025-11-10       112.01    6832.42   0.014859  0.015398  0.917278   
     2025-11-11       113.64    6846.62   0.014552  0.002078  0.915227   
     2025-11-12       115.75    6850.93   0.018567  0.000630  0.915819   
     2025-11-13       115.42    6737.48  -0.002851 -0.016560  0.910274   
     2025-11-14       111.22    6734.10  -0.036389 -0.000502  0.913214   
     2025-11-17       110.29    6672.42  -0.008362 -0.009159  0.913563   
     2025-11-18       114.09    6617.33   0.034455 -0.008256  0.905770   
     2025-11-19       110.00    6642.15  -0.035849  0.003751  0.900194   
     2025-11-20       105.67    6538.77  -0.039364 -0.015564  0.909773   
     2025-11-21       104.31    6602.98  -0.012870  0.009820  0.901516   
     2025-11-24       106.97    6705.11   0.025501  0.015467  0.906345   
     2025-11-25       104.40    6765.89  -0.024025  0.009065  0.897253   
     2025-11-26       106.14    6812.60   0.016667  0.006904  0.899291   

                 beta_750  beta_1250    rf_1y     rf_3y    rf_5y   E_Rm_1y  \
AVGO date                                                                    
NFLX 2025-10-30  1.122784   1.288617  0.03757  0.037385  0.03720  0.176047   
     2025-10-31  1.122790   1.296852  0.03718  0.037175  0.03717  0.177048   
     2025-11-03  1.130791   1.297005  0.03783  0.037490  0.03715  0.182072   
     2025-11-04  1.132288   1.295684  0.03793  0.037470  0.03701  0.188947   
     2025-11-05  1.131534   1.296016  0.03788  0.037765  0.03765  0.188508   
     2025-11-06  1.126597   1.297353  0.03763  0.037260  0.03689  0.180142   
     2025-11-07  1.114149   1.299174  0.03757  0.037185  0.03680  0.169138   
     2025-11-10  1.108177   1.298943  0.03783  0.037465  0.03710  0.159243   
     2025-11-11  1.113979   1.300530  0.03783  0.037510  0.03719  0.153890   
     2025-11-12  1.111048   1.300523  0.03778  0.037230  0.03668  0.150763   
     2025-11-13  1.106794   1.299131  0.03793  0.037480  0.03703  0.133234   
     2025-11-14  1.105667   1.300518  0.03788  0.037615  0.03735  0.135622   
     2025-11-17  1.107360   1.301190  0.03772  0.037465  0.03721  0.126230   
     2025-11-18  1.101492   1.298513  0.03772  0.037330  0.03694  0.124027   
     2025-11-19  1.101657   1.297460  0.03772  0.037400  0.03708  0.140978   
     2025-11-20  1.106397   1.299810  0.03775  0.037240  0.03673  0.121499   
     2025-11-21  1.102880   1.297115  0.03740  0.036790  0.03618  0.127356   
     2025-11-24  1.104574   1.297697  0.03730  0.036675  0.03605  0.142797   
     2025-11-25  1.100427   1.295456  0.03732  0.036485  0.03565  0.146525   
     2025-11-26  1.078608   1.296374  0.03732  0.036485  0.03565  0.149961   

                  E_Rm_3y   E_Rm_5y     Re_1y     Re_3y     Re_5y  
AVGO date                                                          
NFLX 2025-10-30  0.212604  0.148410  0.163303  0.234118  0.180508  
     2025-10-31  0.217039  0.146579  0.164316  0.239125  0.179057  
     2025-11-03  0.213043  0.147209  0.168771  0.236003  0.179896  
     2025-11-04  0.205868  0.143299  0.176413  0.228145  0.174727  
     2025-11-05  0.205216  0.146048  0.176080  0.227241  0.178136  
     2025-11-06  0.208440  0.141050  0.168020  0.230111  0.172022  
     2025-11-07  0.190239  0.138957  0.158004  0.207710  0.169519  
     2025-11-10  0.192308  0.143027  0.149200  0.209058  0.